In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

groq = init_chat_model("groq:qwen/qwen3.6-27b")
response = groq.invoke("Hi who are you?")
response

AIMessage(content='\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - User says: "Hi who are you?"\n   - This is a straightforward greeting and identity question.\n\n2.  **Identify Key Requirements:**\n   - Acknowledge the greeting ("Hi")\n   - State my identity clearly and accurately\n   - Keep it concise and friendly\n   - Align with my actual identity as Qwen (通义千问)\n\n3.  **Formulate Response:**\n   - Greeting: "Hi there!"\n   - Identity: "I\'m Qwen, a large language model developed by Alibaba Group\'s Tongyi Lab."\n   - Offer help: "How can I assist you today?"\n   - Keep it natural and conversational.\n\n4.  **Self-Correction/Verification:**\n   - Check against guidelines: I should identify as Qwen / 通义千问, developed by Alibaba Group\'s Tongyi Lab.\n   - The response matches this exactly.\n   - Tone is friendly and helpful.\n   - No extra fluff or false claims.\n\n   All good. Proceed. \n   Output matches the refined response.✅\n</think>\n\nHi there! I\'m 

In [2]:
# Method 1 of tool calling
from langchain.tools import tool

# Using this decorator marks the function as a tool
@tool
def get_weather(city: str) -> str:
    """Get the weather for the particular city."""
    return f"It's sunny in {city}"


qwenModel_withTool = groq.bind_tools([get_weather])

In [3]:
# Method 2
from langchain.agents import create_agent

agent = create_agent(
    model="groq:qwen/qwen3-32b",
    system_prompt="You are a helpfull assistance",
    tools=[get_weather],
)

In [4]:
# How to call the tool?

response = qwenModel_withTool.invoke("What's the weather in NYC now?")
print(response)

content='' additional_kwargs={'reasoning_content': 'Thinking Process:\n1.  Identify user intent: User wants to know the current weather in NYC (New York City).\n2.  Identify available tool: `get_weather` function takes a `city` parameter.\n3.  Extract parameter: `city` = "NYC" or "New York City".\n4.  Call tool: `get_weather(city="NYC")`.\n5.  Process response: Return the weather information to the user.\n6.  Formulate response: State the weather clearly based on the tool\'s output. (I will simulate the tool call and then generate the response based on expected output, but since I am an AI, I will just call the tool.)\n\nWait, I need to output the tool call exactly as specified.\nTool: `get_weather`\nParameter: `city`: "NYC"\nLet\'s generate the tool call.✅\n', 'tool_calls': [{'id': '114kep5q6', 'function': {'arguments': '{"city":"NYC"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 217, 'prompt_tokens': 277, 'total_tokens': 494,

# Tool Execution Loop

## Example

### User

> "What's the weather in Delhi tomorrow, and should I carry an umbrella?"

The LLM cannot know tomorrow's weather.

Instead, it returns something like:

```json
{
  "tool": "weather",
  "arguments": {
    "city": "Delhi",
    "day": "tomorrow"
  }
}
```

Your code executes:

```python
weather_tool(city="Delhi", day="tomorrow")
```

Result:

```json
{
  "temperature": 29,
  "rain": true
}
```

You send this result back to the LLM.

The LLM now answers:

> Tomorrow will be around **29°C** with rain expected. Yes, you should carry an umbrella.

---

## Why is it called a "loop"?

Because one tool often isn't enough.

### Example

> Find the latest OpenAI GPT-6 announcement and summarize it.

The execution flow might look like this:

```text
LLM
 │
 ▼
Search Web
 │
 ▼
LLM reads search results
 │
 ▼
Open webpage
 │
 ▼
LLM reads article
 │
 ▼
Summarize
 │
 ▼
Done
```

Multiple tool calls can happen before the final answer is generated.

---

## A Simple Implementation

```python
messages = [{"role": "user", "content": user_prompt}]

while True:
    response = llm.chat(messages)

    if response.has_tool_call():
        result = execute_tool(
            response.tool_name,
            response.arguments
        )

        messages.append(response)
        messages.append({
            "role": "tool",
            "content": result
        })
    else:
        print(response.text)
        break
```

This is the basic pattern used in many AI agent frameworks.

---

## Visual Representation

```text
User
  │
  ▼
LLM
  │
Needs tool?
  │
 Yes
  ▼
Execute Tool
  │
  ▼
Tool Result
  │
  ▼
LLM
  │
Need another tool?
  │
 Yes ───────────────┐
                    │
                    ▼
             Execute Tool
                    │
                    ▼
                Tool Result
                    │
                    ▼
                  LLM
                    │
                    ▼
             Final Response
```

---

## Why It's Important

Without a tool execution loop, an LLM can only answer using the knowledge it already has.

With a tool execution loop, it can:

- Search the web
- Read files
- Query databases
- Execute Python code
- Send emails
- Call APIs
- Use a calculator
- Access internal company data

The LLM becomes capable of interacting with external systems instead of only generating text.

